# Corneal-FM v6.0 — End-to-End Walkthrough

> **对标**: eye-scGPT Protocol (Ding et al. 2026, *Nature Protocols*)
> **管线**: 数据准备 → Fine-tune → 零样本推理 → Benchmark → 发布
> **模型**: Geneformer · scGPT · scLong · tahoe-x1 · stack

This notebook walks through the complete Corneal-FM pipeline for tissue-specific
fine-tuning of single-cell foundation models on the human cornea.

## 0. Environment Setup

```bash
conda env create -f protocol_env.yml
conda activate corneal-fm
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import scanpy as sc
import torch
from corneal_fm.config import load_config, set_seed

set_seed(42)
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Project: {PROJECT_ROOT}")

## 1. Data Preparation

### 1.1 Download CELLxGENE Ocular Surface Atlas

The training data is the publicly available Human Ocular Surface Atlas (815,769 cells)
from CELLxGENE Discover.

```bash
python protocol_01_subsample.py  # Subsample 100k train + 20k test
```

In [ ]:
# Verify training data
train = sc.read_h5ad("data/corneal_100k_subsample.h5ad")
test = sc.read_h5ad("data/corneal_20k_test.h5ad")
print(f"Train: {train.n_obs:,} cells × {train.n_vars:,} genes")
print(f"Test:  {test.n_obs:,} cells × {test.n_vars:,} genes")
print(f"Cell types: {train.obs['cell_type'].nunique()}")

### 1.2 Prepare scGPT-compatible data

```bash
python protocol_02_prep_scgpt.py  # Top 5,000 HVGs + gene vocab matching
```

### 1.3 Cross-species ortholog mapping

```bash
python protocol_03_orthologs.py  # Ensembl Compara 1:1 orthologs
```

### 1.4 Tokenize for Geneformer

```bash
python protocol_04_tokenize_gf.py  # Rank-value encoding → HF dataset
```

In [ ]:
# Verify tokenized data
from datasets import load_from_disk
ds = load_from_disk("data/gf_tokenized_train/corneal_100k.dataset")
print(f"Tokenized cells: {len(ds):,}")
print(f"Columns: {ds.column_names}")

### 1.5 CellRefine Marker Gene Prototypes (v6.0 A3)

```bash
python protocol_10_cellrefine_markers.py --output data/cellrefine_prototypes.json
```

In [ ]:
import json
with open("data/cellrefine_prototypes.json") as f:
    prototypes = json.load(f)
print(f"Cell types with prototypes: {prototypes['n_cell_types']}")
print(f"Lineage edges: {prototypes['n_lineage_edges']}")

# Show example: corneal superficial markers (fine → coarse)
cs = prototypes["prototypes"]["corneal_superficial"]
print(f"\ncorneal_superficial markers: {' → '.join(cs['genes'])}")

## 2. Model Fine-tuning

### 2.1 Geneformer Fine-tune (Corneal-GF)

```bash
python protocol_06_finetune_gf.py --config config/default_train_args.yml
```

**Expected**: 2-3 days on GX10 (128 GB unified memory)

In [ ]:
# Load fine-tuned model (after training completes)
from corneal_fm.trainer import GeneformerTrainer

trainer = GeneformerTrainer.load("save/gf_corneal_202607XX-XXXXXX")
print(f"Best val loss: {trainer.best_val_loss:.4f}")

### 2.2 scGPT Fine-tune (Corneal-scGPT)

```bash
python protocol_07_finetune_scgpt.py --config config/default_train_args.yml
```

**Expected**: 3-5 days on GX10

### 2.3 Ablation Studies

| Ablation | Command | GPU Time |
|----------|---------|----------|
| A1 Data Efficiency | `--ablation data_efficiency --sizes 1k,5k,10k,25k,50k,100k` | 3-5 days |
| A2 Layer Freezing | `--ablation freeze_layers --options 0,2,4,6` | 2-3 days |
| A3 CellRefine | `--ablation cellrefine --prototypes data/cellrefine_prototypes.json` | +1 epoch |

## 3. Zero-Shot Inference

### 3.1 scLong (1B parameters)

```bash
python protocol_08_inference_sclong.py
```

### 3.2 tahoe-x1 (v6.0)

```bash
python protocol_08b_inference_tahoe.py
```

### 3.3 stack — In-Context Learning (v6.0)

```bash
python protocol_08c_inference_stack.py
```

## 4. Benchmark

```bash
python protocol_09_benchmark.py
```

### Evaluation Tasks

| Task | Metric | Clinical Question |
|------|--------|-------------------|
| **E1** Cell Classification | Macro F1, Accuracy | Can FM identify corneal cell types? |
| **E2** Perturbation Prediction | Pearson r, FDR | Can in silico KO predict DED targets? |
| **E3** Cross-Species (CSPS) | Accuracy retention | Do human models transfer to mouse/rat? |
| **E4** Embedding Analysis | Silhouette, Centroid dist | Does latent space separate normal vs DED? |

In [ ]:
# Load benchmark results
import pandas as pd
results = pd.read_csv("save/benchmark_results.csv")
results.pivot_table(
    index="task", columns="model", values="score", aggfunc="mean"
).style.background_gradient(cmap="Blues")

## 5. Model Release

```bash
python protocol_11_release.py --model_dir save/gf_corneal_20260701-120000
python protocol_11_release.py --model_dir save/scgpt_corneal_20260701-120000
```

Creates self-contained `.zip` archives with:
- `best_model.pt` — Fine-tuned weights
- `gene_vocab.json` — Gene vocabulary
- `train_args.yml` — Training hyperparameters
- `run.log` — Training log
- `README.md` — Usage instructions

## 6. Full Pipeline (One Command)

```bash
python run_all.py --config config/default_train_args.yml
```

Or dry-run to verify:
```bash
python run_all.py --dry_run
```